# Dimensional — Star Schema Query

Join `globalmart.gold.fact_sales` to all four dimensions. Filter to **2018**, states **SP / RJ / MG**, and **current** customer versions. Aggregate by month, state, and product category; show top **20** groups by revenue.

In [ ]:
import sys
import importlib

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
REPO_ROOT = notebook_path.split("/notebooks/")[0]
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import src.dimensional.star_schema_query as star_module
importlib.reload(star_module)

from src.dimensional.star_schema_query import (
    StarSchemaQueryConfig,
    build_star_schema_summary,
    run_star_schema_query,
)
from src.ingestion.idempotent_loader import save_run_report_to_volume

config = StarSchemaQueryConfig(filter_year=2018, filter_states=("SP", "RJ", "MG"), top_n=20)
print("Loaded from:", star_module.__file__)

In [ ]:
summary = build_star_schema_summary(spark, config)
print("Summary groups:", summary.count())
display(summary.limit(20))

In [ ]:
import json

report = run_star_schema_query(spark, config)
print(json.dumps(report, indent=2, default=str))
try:
    save_run_report_to_volume(
        report,
        dbutils,
        "/Volumes/globalmart/metadata/run_reports/dimensional_star_schema_query.json",
    )
except Exception as exc:
    print("Volume save skipped:", exc)